# Part 1. 왜 Agent를 평가하는가

이 노트북은 Day05_new의 두 번째 노트북입니다. 에이전트가 내놓은 **그럴듯한 답변**이 왜 업무가 성공했다는 증거가 되지 못하는지를, 주문 취소 업무 하나로 직접 확인합니다. 같은 최종 답변을 내는 두 실행을 나란히 놓고, 답변만 보는 평가가 무엇을 놓치는지 눈으로 봅니다. 그다음 같은 입력을 반복해서 돌렸을 때 결과가 흔들린다는 사실(비결정성)을 pass@k와 pass^k로 읽고, 마지막으로 "점수가 낮다"와 "평가할 수 없다"가 서로 다른 말이라는 점을 구분합니다.

### 수업 시간표

| 구간 | 내용 | 분 |
|---|---|---|
| 0. 준비 | 가져오기, 스위치, dev 사례와 fixture 읽기 | 3 |
| 1. 답변, 결과, 과정 | 한 번의 실행을 세 가지 시선으로 나눠 읽기 | 6 |
| 2. 같은 답, 다른 궤적 | 승인 전 쓰기를 섞은 실행과 정상 실행 비교 | 12 |
| 3. 비결정성 | 반복 실행, pass@k와 pass^k | 9 |
| 4. 낮은 점수와 평가 불능 | FAIL과 INCOMPLETE, PROVISIONAL, NO_EVIDENCE | 6 |
| 5. 직접 해보기 | 내 실수 조합 만들기, 반복 늘리기, 실제 모델 한 번 | 4 |

### 이 노트북이 하는 일 (그리고 하지 않는 일)

- 하는 일: `eval_lab`의 업무 환경(SQLite로 만든 가상 주문·결제·재고 서비스)에서 **scripted** 실행을 돌립니다. scripted는 모델이 없는 실행기입니다. 도구를 정해진 순서로 실제로 호출하고, 일부러 실수를 끼워 넣을 수 있습니다. 그 실행 기록(근거)을 두 가지 평가기, 즉 기대 상태와 비교하는 오프라인 평가기(`eval_lab.evaluators.evaluate`)와 정답 없이 궤적만 보는 규칙 검사(`eval_lab.trajectory_rules.evaluate_trajectory`)로 채점합니다.
- 하지 않는 일: **scripted 결과는 LLM 성능이 아닙니다.** 이 노트북의 반복 실행, 실수 확률, 통과율은 모두 *모의 실행*이 만든 숫자입니다. 모델이 이런 실수를 얼마나 자주 하는지는 이 노트북이 말해 주지 않습니다. 실제 모델은 직접 해보기의 선택 셀에서 Episode 하나만 돌려 봅니다. 그 결과도 한 번의 관찰일 뿐 성능이 아닙니다.
- 평가기를 믿는 방법(Golden Dataset, 평가기 평가)은 뒤 노트북의 일입니다. 여기서는 "무엇을 재야 하는가"까지만 갑니다.

### 호출 규모

- 0~4절의 모든 셀: 유료 호출이 없습니다. Claude도 Jev도 Judge도 부르지 않습니다. 모델 없는 scripted 실행과 코드 평가기만 씁니다. 3절의 반복 실행은 Episode 수십 개이지만 모두 로컬에서 몇 초 안에 끝납니다.
- Langfuse: 키가 설정되어 있으면 2절의 첫 비교(Episode 2개)만 Langfuse에 기록합니다. 키가 없으면 안내 한 줄을 출력하고 기록 없이 진행합니다. 모델 호출이 아니라서 비용은 없지만 데이터는 외부(Langfuse)로 갑니다. 모든 데이터는 교육용 가상 데이터입니다.
- 직접 해보기의 선택 셀 하나(`RUN_OPTIONAL`): Claude 에이전트 Episode 한 개, 호출은 최대 12회, 1분 안팎입니다(Episode당 모델 호출 상한이 12회로 묶여 있습니다). 기본값은 꺼져 있고, 켜면 실행 직전에 셀이 같은 규모를 한 줄로 다시 알립니다. 이 비용은 본인의 Anthropic 크레딧에서 나갑니다. Jev와 Judge는 부르지 않습니다.

### 표지 규칙

코드 셀 첫 줄의 표지는 하는 일을 알려 줍니다. `# [따라하기]`는 그대로 실행, `# [바꿔보기]`는 값을 바꿔 다시 실행, `# [깨뜨리기]`는 일부러 평가를 속이거나 빈틈을 찾는 셀, `# [내 것 만들기]`는 내 조합을 직접 쓰는 셀입니다. **먼저 예측하세요.** 가 나오면 실행하기 전에 결과를 한 줄이라도 적어 보세요. 맞히는 것이 목적이 아니라, 틀렸을 때 어느 가정이 틀렸는지 찾는 것이 목적입니다.

## 0. 준비: 사례와 도구

주문 취소 업무는 이렇게 생겼습니다. 고객이 "이 주문을 취소하고 환불해 주세요"라고 하면 에이전트는 주문을 조회하고(`get_order`), 취소 정책을 확인하고(`check_cancel_policy`), 취소를 시작하고(`begin_cancellation`), 결제를 조회해 환불하고(`get_payment`, `refund_payment`), 재고를 되돌리고(`release_stock`), 취소를 완료한 뒤(`complete_cancellation`) 상태를 다시 조회(`get_order`)해서 고객에게 알립니다. 결제 서비스가 TIMEOUT을 내거나 이미 출고된 주문이거나 다른 고객의 주문이면 흐름이 달라집니다.

이 노트북이 쓰는 사례는 `data/scenarios.json`의 **dev 사례** 아홉 개입니다. 사례마다 초기 상태(fixture)와 평가자 전용 기대 명세가 있고, 에이전트에게는 요청과 도구만 갑니다. 평가자 전용 기대 명세(`expected_output`)는 에이전트 쪽 코드로 넘어가지 않습니다. 아래 셀은 사례를 읽고, 이 노트북 내내 쓸 도우미 두 개를 만듭니다.

- `run_episode`: 사례 하나를 scripted로 한 번 실행해서 근거(Artifact)를 돌려줍니다. 드라이버(`driver`)를 주지 않으면 기본 드라이버가, 주면 그 드라이버가 도구를 호출합니다.
- 스위치 두 개: `RUN_EXERCISE`와 `RUN_OPTIONAL`은 직접 해보기에서만 쓰며 기본값은 둘 다 `False`입니다.

In [ ]:
# [따라하기] 가져오기, 스위치, 작업 폴더
import json
from collections import Counter

from eval_lab import settings as lab_settings
from eval_lab.agent import PROMPTS
from eval_lab.dataset import load_dataset, load_fixtures
from eval_lab.episodes import episode_summary, execute_episode
from eval_lab.evaluators import evaluate, state_summary
from eval_lab.notebook import announce, langfuse_or_none, new_run_dir, show
from eval_lab.reporting import compare
from eval_lab.trajectory_rules import RULES, evaluate_trajectory
from eval_lab.traffic import make_mistake_driver

# 직접 해보기에서만 쓰는 스위치. 둘 다 꺼진 채로 시작한다.
RUN_EXERCISE = False  # True면 해보기 2의 큰 반복(여전히 모델 호출 없음)을 돌린다
RUN_OPTIONAL = False  # True면 해보기 3에서 실제 Claude Episode 하나를 돌린다(유료)

# 이 노트북이 파일을 쓰는 곳은 이 폴더 하나뿐이다.
run_dir = new_run_dir("01")
print("작업 폴더:", run_dir)

In [ ]:
# [따라하기] dev 사례 아홉 개와 fixture를 읽고, 사례 하나를 실행하는 도우미를 만든다
# 1단계: 승인된 dev 분할만 읽는다. 기대 명세(expected_output)는 평가 단계에서만 쓴다.
cards = {card.metadata.scenario_id: card for card in load_dataset(split="dev")}
fixtures = load_fixtures()

show([{"사례": sid, "분류": card.metadata.slice, "주문 번호 있음": card.input.order_id is not None,
       "기대 응답 유형": card.expected_output.response_kind} for sid, card in cards.items()],
     title="dev 사례 (승인됨)")


# 2단계: 사례 하나를 scripted로 한 번 실행한다. 모델 호출은 없다.
def run_episode(card, *, driver=None, release="baseline", trial=1, langfuse=None, version=None, tags=None, max_turns=None):
    """scripted Episode 하나. driver가 없으면 기본 드라이버, 있으면 그 드라이버가 도구를 호출한다."""
    return execute_episode(
        card.input, fixtures[card.fixture_id], followups=card.followups,
        max_turns=card.max_turns if max_turns is None else max_turns,
        scenario_id=card.metadata.scenario_id, release=release, trial=trial,
        mode="scripted", prompt=PROMPTS[release], scripted_driver=driver,
        langfuse=langfuse, environment_tag="day05_new", tags=tags, version=version)


def verdict_rows(artifact, card):
    """오프라인 평가기(기대 상태와 비교)의 검사별 결과."""
    return [{"검사": v.name, "상태": v.status, "이유": v.reason}
            for v in evaluate(artifact, card.expected_output.model_dump())]


def result_label(result):
    """도구 결과를 한 칸 문자열로 줄인다."""
    label = result.get("code") or result.get("status") or "-"
    if "allowed" in result:
        label += f" (allowed={result['allowed']})"
    return label


def trace_rows(artifact):
    """도구 호출 순서(궤적)를 표로 만든다."""
    return [{"단계": e.step, "도구": e.tool, "결과": result_label(e.result)} for e in artifact.events]

## 1. 답변, 결과, 과정: 세 가지 질문

에이전트가 일을 끝냈다고 말할 때 우리가 알고 싶은 것은 사실 서로 다른 세 가지입니다.

| 질문 | 이름 | 무엇을 보나 | 근거 |
|---|---|---|---|
| 실제 목표 상태에 도달했는가 | 결과(Outcome) | 주문이 `cancelled`인가, 환불 금액이 맞는가, 재고가 돌아왔는가 | 도구 결과와 업무 환경에서 읽은 상태 |
| 올바른 대상, 권한, 순서로 처리했는가 | 과정(Trajectory) | 정책 확인 전에 쓰기를 했는가, 남의 주문을 건드렸는가 | 도구 호출의 인자와 순서 |
| 실행한 사실을 정확히 전달했는가 | 답변(Final Response) | 고객에게 한 말이 기록과 맞는가 | 최종 답변 문장과 실행 증거 |

세 질문은 서로를 보증하지 않습니다. 답변이 매끄러워도 결과가 틀릴 수 있고(말만 완료), 결과가 맞아도 과정이 위험했을 수 있으며(운이 좋았을 뿐), 과정이 완벽해도 답변이 부풀려질 수 있습니다. 그래서 평가는 "답변이 그럴듯한가"로 시작하지 않고 **무엇을 근거로 무엇을 판정할지**를 먼저 나눕니다.

이 저장소에서 세 질문의 담당자는 이렇게 나뉩니다.

- 결과: `evaluators.check_outcome`(최종 상태), `check_side_effects`(건드리면 안 되는 상태가 그대로인가). 기대 상태가 필요합니다.
- 과정: `trajectory_rules`의 이름 있는 규칙들. 기대 정답 없이 기록만으로 돕니다.
- 답변: `evaluators.check_response`(응답 유형과 구조화한 주장이 실제 상태와 맞는가)가 구조를 검사하고, 자유 문장의 의미는 뒤에서 Jev와 Judge가 맡습니다.

아래 셀은 정상 실행 하나를 세 시선으로 나눠 놓습니다. 정상 실행이라서 셋 다 문제가 없어 보이는 것이 정상입니다. 나중에 문제가 있는 실행과 이 표를 비교합니다.

In [ ]:
# [따라하기] 정상 실행 한 번을 답변, 결과, 과정으로 나눠 읽는다
normal_card = cards["normal"]
normal_run = run_episode(normal_card)

# 답변: 에이전트가 고객에게 한 말
print("답변:", normal_run.response.kind, "|", normal_run.response.message, "| 환불 금액(센트):", normal_run.response.refund_cents)

# 결과: 업무 환경에서 직접 읽은 마지막 상태 (에이전트의 말이 아니다)
print("결과:", state_summary(normal_run.final_state))

# 과정: 서비스가 실제로 받은 도구 호출 순서
show(trace_rows(normal_run), title="과정: 도구 호출 순서")

# 세 시선의 판정 요약
trajectory = evaluate_trajectory(normal_run)
print("과정 판정:", trajectory.status, "-", trajectory.reason)
show(verdict_rows(normal_run, normal_card), title="결과·답변 검사 (기대 상태와 비교)")

**관찰할 것.** 같은 한 번의 실행에서 `답변`은 에이전트가 쓴 문장이고, `결과`는 업무 환경이 알려 준 숫자이며, `과정`은 서비스가 받은 호출 목록입니다. 세 값의 출처가 다릅니다. 답변 한 줄만 로그에 남겨 두었다면 결과와 과정은 영영 복원할 수 없습니다. 이것이 이 저장소가 Episode마다 **근거(Artifact)** 를 따로 모으는 이유입니다. 표에서 `outcome`은 결과, `policy`와 `recovery`는 과정에 가까운 검사, `response_contract`는 답변 구조 검사입니다. 오프라인 평가기 안에도 과정 검사가 일부 들어 있다는 점을 기억해 두세요. 2절에서 이 검사와 궤적 규칙이 어떻게 다른지 봅니다.

## 2. 같은 답, 다른 궤적

주문 번호가 있는 정상 요청에서, 한 에이전트는 정책을 확인한 뒤 취소를 시작합니다. 다른 에이전트는 **정책 확인 없이 먼저 취소를 시작해 보고**, 그 뒤에 정책을 확인하고 나머지를 똑같이 진행합니다. 둘 다 끝에서는 주문이 취소되고 환불과 재고 복원이 끝나며, 고객에게는 같은 문장을 말합니다.

이번 정상 요청에서 먼저 취소를 시작한 쪽은 우연히 정책 확인도 통과하는 주문이었습니다. 만약 그 주문이 출고된 주문이었다면 같은 습관이 잘못된 쓰기를 만들었을 것입니다. 결과만 보면 이번에는 운이 좋았을 뿐입니다. 이것이 "같은 답변이라도 같은 성공은 아니다"의 의미입니다.

두 번째 실행은 `make_mistake_driver`로 만듭니다. 이 드라이버는 정상 흐름을 따라가다가 정해진 실수를 **실제 도구 호출**로 끼워 넣습니다. 말로 적는 실수가 아니라 서비스에 Event로 남는 실수입니다. 여기서는 `write_before_approval`(승인 전 쓰기) 확률을 1.0으로 주어 항상 일어나게 합니다. 이것은 *모의 실행*입니다. 모델이 이 실수를 하는지는 이 셀이 알려 주지 않습니다.

**먼저 예측하세요.** 아래 셀을 실행하기 전에 세 가지를 적어 보세요. (a) 두 실행의 최종 답변과 마지막 업무 상태는 같을까 다를까. (b) 오프라인 평가기(`evaluate`)는 두 실행을 같게 볼까. (c) 궤적 규칙(`evaluate_trajectory`)의 판정은 어떨까.

In [ ]:
# [따라하기] 같은 사례를 정상 드라이버와 승인 전 쓰기 드라이버로 한 번씩 실행한다 (모델 호출 없음)
# Langfuse 키가 있으면 두 Episode만 기록한다. 없으면 안내 한 줄을 내고 기록 없이 진행한다.
langfuse = langfuse_or_none()
if langfuse is not None:
    print("Langfuse에 Episode 2개를 기록합니다(교육용 가상 데이터).")

# 1단계: 정상 실행과 실수 실행. version 이름이 Langfuse의 version 라벨이 된다.
good_run = run_episode(normal_card, langfuse=langfuse, version="driver-default", tags=["01-why-evaluate"])
mistaken_driver = make_mistake_driver({"write_before_approval": 1.0}, seed=0)
bad_run = run_episode(normal_card, driver=mistaken_driver, langfuse=langfuse,
                      version="driver-write-before-approval", tags=["01-why-evaluate"])
if langfuse is not None:
    langfuse.flush()

# 2단계: 답변과 마지막 상태가 같은지 본다.
print("최종 답변이 같은가:", good_run.response == bad_run.response)
print("마지막 업무 상태가 같은가:", state_summary(good_run.final_state) == state_summary(bad_run.final_state))

# 3단계: 도구 호출 순서를 나란히 본다.
show(trace_rows(good_run), title="정상 드라이버의 궤적")
show(trace_rows(bad_run), title="승인 전 쓰기 드라이버의 궤적")

In [ ]:
# [따라하기] 같은 두 실행을 두 평가기로 채점한다: 기대 상태 평가기와 궤적 규칙
good_verdicts = {row["검사"]: row["상태"] for row in verdict_rows(good_run, normal_card)}
bad_verdicts = {row["검사"]: row["상태"] for row in verdict_rows(bad_run, normal_card)}
show([{"검사": name, "정상 실행": good_verdicts[name], "실수 실행": bad_verdicts[name]} for name in good_verdicts],
     title="오프라인 평가기(evaluate): 검사별 상태")

for label, run in (("정상 실행", good_run), ("실수 실행", bad_run)):
    verdict = evaluate_trajectory(run)
    print(f"{label} 궤적 판정: {verdict.status} | {verdict.reason}")
    step_of = {event.event_id: event for event in run.events}
    for hit in verdict.fatal:
        event = step_of.get(hit.event_id)
        where = f"단계 {event.step} {event.tool}" if event else hit.event_id
        print(f"  치명 규칙 {hit.rule_id} @ {where}: {hit.reason}")

# 세 단계로 쪼갠 이 저장소의 규칙 목록 (id, 등급)
show([{"규칙": rule_id, "등급": severity} for rule_id, severity, _ in RULES], title="궤적 규칙 목록")

**관찰할 것.** 최종 답변과 마지막 업무 상태는 두 실행에서 똑같습니다. 그런데 오프라인 평가기에서 `outcome`, `side_effects`, `response_contract`는 모두 통과이고 **`policy` 하나만** 갈립니다. 궤적 규칙은 두 번째 실행을 `FAIL`로 판정하고 `write-before-approval`이라는 이름과 함께 **어느 단계의 어느 도구 호출이 처음 위반이었는지**(단계 2의 `begin_cancellation`)까지 가리킵니다. 승인 전 쓰기는 정책 확인(`check_cancel_policy`)이 허용하기 전에 같은 주문에 쓰기 도구를 부른 것입니다.

두 평가기는 같은 위반을 다른 모양으로 말합니다. 오프라인 평가기는 정답 명세와 비교해 `policy` 한 칸을 FAIL로 만들 뿐 어디가 문제인지 알려 주지 않고, 기대 명세가 있어야 돕니다. 궤적 규칙은 정답 없이 기록만으로 돌고, 이름 있는 규칙과 첫 위반 사건을 줍니다. 운영 중인 에이전트에는 정답이 없으므로 이 차이가 03a, 03b의 출발점이 됩니다.

Langfuse 키가 있다면 두 Episode가 `driver-default`, `driver-write-before-approval`이라는 version 라벨로 기록되어 있습니다. 같은 요청의 두 trace를 나란히 열어 보면 승인 전 쓰기가 호출 순서에서 어떻게 보이는지 확인할 수 있습니다. 키가 없으면 이 부분은 건너뜁니다. 키 없이 진행했다면 위 표가 유일한 근거입니다.

### 모든 dev 사례에 같은 실수를 넣으면

위 결과가 사례 하나의 우연인지 알아봅니다. 아홉 개 dev 사례마다 정상 실행과 승인 전 쓰기 실행을 만들고, 두 질문을 던집니다. "최종 답변 유형과 마지막 상태가 같은가", "궤적 판정은 어떻게 갈리는가".

실수가 아예 일어나지 못하는 사례도 있습니다. 주문 번호가 없거나 다른 고객의 주문이라서 에이전트가 쓰기 전에 멈추면, 드라이버가 실수를 끼워 넣을 자리도 없습니다. 그런 사례에서 판정이 같은 것은 정상입니다. 이 표에는 `release="candidate"`를 씁니다. 결제 TIMEOUT 사례에서 candidate는 결과를 조회해 미확정을 해소하는 경로이므로, 정상 실행의 답이 기대와 맞는 쪽에서 비교할 수 있습니다. scripted의 candidate와 baseline 차이는 이 TIMEOUT 처리뿐입니다.

In [ ]:
# [바꿔보기] 아홉 개 dev 사례에 승인 전 쓰기를 항상 넣어 정상 실행과 비교한다 (모델 호출 없음)
# 드라이버를 바꾸거나 release를 "baseline"으로 바꿔서 다시 실행해 보세요.
scan_rows = []
for sid, card in cards.items():
    normal_scan = run_episode(card, release="candidate")
    mistaken_scan = run_episode(card, release="candidate", driver=make_mistake_driver({"write_before_approval": 1.0}, seed=0))
    scan_rows.append({
        "사례": sid,
        "답변 유형 같음": normal_scan.response.kind == mistaken_scan.response.kind,
        "마지막 상태 같음": state_summary(normal_scan.final_state) == state_summary(mistaken_scan.final_state),
        "정상 궤적": evaluate_trajectory(normal_scan).status,
        "실수 궤적": evaluate_trajectory(mistaken_scan).status,
        "실수 치명 규칙": ", ".join(sorted({hit.rule_id for hit in evaluate_trajectory(mistaken_scan).fatal})) or "-",
    })
show(scan_rows, title="사례별 정상 실행과 실수 실행")
print("실수가 궤적을 FAIL로 바꾼 사례:", sum(row["정상 궤적"] != row["실수 궤적"] for row in scan_rows), "/", len(scan_rows))

**관찰할 것.** `답변 유형 같음`과 `마지막 상태 같음`이 모두 `True`인데 `실수 궤적`이 `FAIL`인 줄이 여러 개입니다. 반대로 주문 번호가 없거나 다른 고객의 주문인 사례처럼 실수가 일어날 자리가 없는 줄은 궤적도 같습니다. 일부 사례에서는 `실수 치명 규칙` 칸에 규칙이 둘 이상 보입니다. 이미 취소되었거나 출고된 주문에 먼저 쓰기를 시도하면 `write-after-forbidden` 같은 연쇄 위반도 함께 잡히기 때문입니다. 한 번의 실수가 한 가지 위반만 만든다고 가정하면 안 됩니다.

### 답변만 보는 평가기는 얼마나 속는가

이제 일부러 나쁜 평가기를 만듭니다. "최종 답변 유형이 기대와 같으면 통과"라는, 답변만 보는 평가기입니다. 실무에서 흔한 첫 평가가 이 모양입니다(로그에 답변 한 줄만 남기기 때문입니다). 같은 열여덟 개 실행(정상 아홉, 실수 아홉)에 이 평가기를 적용하고, 궤적 판정과 비교합니다.

In [ ]:
# [깨뜨리기] 답변 유형만 보는 평가기는 승인 전 쓰기를 얼마나 놓치는가
def answer_only_passes(artifact, card):
    """답변 유형이 기대와 같으면 통과. 상태도 과정도 보지 않는다."""
    return artifact.response is not None and artifact.response.kind == card.expected_output.response_kind


tallies = {"정상": Counter(), "실수": Counter()}
for sid, card in cards.items():
    for label, driver in (("정상", None), ("실수", make_mistake_driver({"write_before_approval": 1.0}, seed=0))):
        run = run_episode(card, release="candidate", driver=driver)
        tallies[label]["답변만 통과"] += answer_only_passes(run, card)
        tallies[label]["궤적 PASS"] += evaluate_trajectory(run).status == "PASS"
        tallies[label]["궤적 FAIL"] += evaluate_trajectory(run).status == "FAIL"

show([{"실행 묶음": label, "사례 수": len(cards), **dict(count)} for label, count in tallies.items()],
     title="답변만 본 평가기와 궤적 규칙의 판정 수")
missed = tallies["실수"]["답변만 통과"] - tallies["실수"]["궤적 PASS"]
print("답변만 보면 통과인데 궤적으로는 PASS가 아닌 실수 실행:", missed)

**관찰할 것.** 실수 실행 묶음에서 `답변만 통과`는 `정상` 묶음과 같은 숫자인데 `궤적 FAIL`이 따로 나옵니다. 답변만 보는 평가기는 실수를 넣든 안 넣든 같은 점수를 줍니다. 이 평가기는 실수를 **측정할 수 없습니다**. 측정할 수 없는 것을 개선할 수도 없습니다. 에이전트를 고쳐서 "답변 점수가 그대로"라고 해도 과정은 나빠졌을 수 있고 좋아졌을 수도 있는데, 그 평가기는 어느 쪽인지 말해 주지 못합니다.

이 셀의 숫자는 모두 모의 실행입니다. 실제 모델이 승인 전 쓰기를 하는 비율은 여기서 알 수 없습니다.

### 글자는 행동이 아니다: 말만 완료

과정과 결과가 같은 것은 아닙니다. 이번에는 결과가 틀린 경우입니다. `claim_without_record`는 취소를 완료하거나 확인하지 않은 채 "완료했습니다"라고 보고하는 실수입니다.

**먼저 예측하세요.** 이 실수를 넣은 실행에서 답변 유형은 무엇일까요? 마지막 업무 상태(`order_status`)는 `cancelled`일까요? 오프라인 평가기와 궤적 규칙은 각각 무엇을 FAIL로 보고할까요?

In [ ]:
# [깨뜨리기] 기록 없는 완료 주장: 답변은 완료인데 업무 상태는 완료가 아니다 (모델 호출 없음)
claim_driver = make_mistake_driver({"claim_without_record": 1.0}, seed=0)
claim_run = run_episode(normal_card, driver=claim_driver)

print("답변:", claim_run.response.kind, "|", claim_run.response.message)
print("실제 마지막 상태:", state_summary(claim_run.final_state))
show(verdict_rows(claim_run, normal_card), title="오프라인 평가기: 검사별 상태")
claim_verdict = evaluate_trajectory(claim_run)
print("궤적 판정:", claim_verdict.status, "|", claim_verdict.reason)
for hit in claim_verdict.fatal:
    print(f"  치명 규칙 {hit.rule_id} @ {hit.event_id}: {hit.reason}")

**관찰할 것.** 답변은 `completed`인데 실제 상태는 `cancelling`에 머물러 있고, `outcome`과 `response_contract`가 FAIL, 궤적은 `claim-without-record`로 FAIL입니다. 이 규칙은 응답의 완료 주장을 **기록된 도구 결과**(여기서는 `get_order`가 `cancelled`를 본 사건)와 대조합니다. 위반 사건 칸이 도구 사건 ID가 아니라 `response`라고 적혀 있는 것도 눈여겨보세요. 문제의 근거가 도구 호출이 아니라 응답 자체이기 때문입니다.

정리하면 승인 전 쓰기는 **과정만** 나쁜 경우이고, 기록 없는 완료는 **답변과 결과** 사이가 어긋난 경우입니다. 세 가지 질문을 나눠 두지 않으면 어느 쪽에서 틀렸는지 말할 수 없습니다.

## 3. 비결정성: 한 번 통과는 증거가 아니다

실제 에이전트는 같은 요청에도 매번 같은 경로를 걷지 않습니다. 도구 선택이 달라지고, 순서가 달라지고, 가끔 틀립니다. 그래서 "한 번 돌려 봤더니 통과"는 평가가 아닙니다. 반복 실행에서는 두 가지 질문이 서로 다른 숫자를 줍니다.

- **pass@k**: k번 시도해서 **한 번이라도** 통과한 사례의 비율. "되기는 하는가, 한 번쯤은 성공하는가"를 묻습니다. 사람이 여러 번 시켜 보고 골라 쓸 수 있는 도구에 어울리는 질문입니다.
- **pass^k**: k번 시도해서 **모두** 통과한 사례의 비율. "항상 되는가"를 묻습니다. 사람이 매번 확인하지 않고 업무를 맡기는 에이전트에 어울리는 질문입니다. 주문 취소처럼 한 번의 잘못된 환불이 돌이킬 수 없는 업무라면 이쪽이 중요합니다.

k가 커질수록 pass@k는 올라가고 pass^k는 내려갑니다. 같은 실행에서 나온 두 숫자가 정반대로 움직입니다. 이 저장소는 `eval_lab.reporting.compare`가 사례 단위로 두 값(`pass_at_k`, `pass_all_k`)을 계산합니다.

**이 절의 실행은 전부 모의입니다.** 모델이 아니라 `make_mistake_driver`가 요청마다 정해진 확률로 승인 전 쓰기를 섞습니다. 실수 선택은 (seed, 요청 ID)로 정해지므로, 회차마다 seed를 다르게 주면 *회차마다 다른 우연*이 만들어집니다. 실제 모델의 비결정성을 흉내 낸 것일 뿐, 모델의 실제 변동폭이 아닙니다.

구성은 이렇습니다. 사례는 실수가 일어날 수 있는 다섯 개(`normal`, `shipped`, `payment-down`, `duplicate-request`, `shipping-race`)이고, 회차는 5회입니다. 통과는 오프라인 평가기의 검사가 **전부** 통과인 경우입니다(`reporting.status_of`의 PASS). `baseline` 칸은 실수 확률 0.3, `candidate` 칸은 실수 없음으로 둡니다. 이 칸 이름은 `compare`가 두 묶음을 요구해서 빌려 쓴 것이고, 여기서는 "흔들리는 에이전트"와 "안정된 에이전트"의 모의입니다. 이 다섯 사례에는 TIMEOUT이 없으므로 `baseline`과 `candidate`의 실행 경로 차이도 없습니다.

**먼저 예측하세요.** 실수 확률 0.3인 쪽의 사례별 한 회차 통과율은 약 몇 %일까요? 그러면 5회 중 한 번이라도 통과하는 사례의 비율(pass@5)과 5회 모두 통과하는 사례의 비율(pass^5)은 각각 대략 얼마일까요? (한 사례의 통과율을 p라 하면 "모두 통과"는 p를 다섯 번 곱한 값에 가깝습니다.)

In [ ]:
# [따라하기] 5개 사례 x 5회 반복 (모의): 흔들리는 쪽과 안정된 쪽의 pass@5, pass^5
REPEAT_SCENARIOS = ["normal", "shipped", "payment-down", "duplicate-request", "shipping-race"]
TRIALS = 5


def collect(release, rate, *, scenario_ids=REPEAT_SCENARIOS, trials=TRIALS):
    """한 묶음의 계획(plan)과 채점된 행(rows)을 만든다. 회차마다 seed가 다르다."""
    plan, rows = [], []
    for sid in scenario_ids:
        card = cards[sid]
        for trial in range(1, trials + 1):
            driver = make_mistake_driver({"write_before_approval": rate}, seed=trial)
            artifact = run_episode(card, driver=driver, release=release, trial=trial)
            verdicts = evaluate(artifact, card.expected_output.model_dump())
            plan.append({"release": release, "scenario_id": sid, "trial": trial, "slice": card.metadata.slice})
            rows.append({"artifact": artifact.model_dump(mode="json"), "verdicts": [v.model_dump() for v in verdicts]})
    return plan, rows


def repeated_summary(flaky_rate):
    """baseline 칸은 flaky_rate, candidate 칸은 실수 없음으로 compare를 부르고 두 칸의 요약을 돌려준다."""
    flaky_plan, flaky_rows = collect("baseline", flaky_rate)
    steady_plan, steady_rows = collect("candidate", 0.0)
    report = {"plan": flaky_plan + steady_plan, "rows": flaky_rows + steady_rows, "mode": "scripted"}
    return compare(report)["summaries"]


summaries = repeated_summary(0.3)
show([{"묶음": name, "계획 수": s["planned"], "통과": s["passed"], "한 회차 통과율": round(s["pass_rate"], 2),
       "pass@5": s["pass_at_k"], "pass^5": s["pass_all_k"]}
      for name, s in summaries.items()],
     title="모의 반복 실행 요약 (사례 5 x 회차 5)")
(run_dir / "repeated_summary.json").write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding="utf-8")

**관찰할 것.** 흔들리는 쪽의 한 회차 통과율은 1보다 꽤 낮은데 `pass@5`는 1.0에 가깝고 `pass^5`는 훨씬 낮습니다. 같은 스물다섯 번의 실행에서 "된다"와 "항상 된다"가 전혀 다른 숫자입니다. 안정된 쪽은 둘 다 1.0입니다. 평가 보고서가 `pass@k` 하나만 싣고 있다면, 매번 확인할 사람이 없는 운영에서는 낙관적인 숫자를 읽고 있는 것일 수 있습니다.

`한 회차 통과율`(`pass_rate`)은 사례와 회차를 가리지 않고 모두 합친 비율입니다. 이 숫자만으로는 "어떤 사례가 늘 실패하는지"와 "모든 사례가 가끔 실패하는지"를 구분하지 못합니다. 사례 단위의 `pass^k`는 후자를 드러냅니다. 이 저장소가 holdout 확인에서 사례 단위로 보는 이유도 같습니다.

### 실수 확률을 바꾸면

실수 확률을 0에서 0.5까지 바꾸면 한 회차 통과율, pass@5, pass^5가 어떻게 움직이는지 봅니다. 오른쪽 끝 열 `통과율^5`은 사례마다 통과율이 같고 회차가 서로 독립이라고 가정했을 때의 어림값입니다. 실제 pass^5와 다를 수 있습니다. 사례가 다섯 개뿐이라 우연의 영향이 크고, 사례마다 실수에 걸리는 회차가 서로 다르기 때문입니다.

In [ ]:
# [바꿔보기] 실수 확률을 바꾸며 한 회차 통과율, pass@5, pass^5를 비교한다 (모의, 모델 호출 없음)
# RATES를 바꾸거나 TRIALS를 3이나 10으로 바꿔서 다시 실행해 보세요.
RATES = (0.0, 0.1, 0.3, 0.5)
steady_plan, steady_rows = collect("candidate", 0.0)
sweep = []
for rate in RATES:
    flaky_plan, flaky_rows = collect("baseline", rate)
    report = {"plan": flaky_plan + steady_plan, "rows": flaky_rows + steady_rows, "mode": "scripted"}
    flaky = compare(report)["summaries"]["baseline"]
    sweep.append({"실수 확률": rate, "한 회차 통과율": round(flaky["pass_rate"], 2),
                  "pass@5": flaky["pass_at_k"], "pass^5": flaky["pass_all_k"],
                  "통과율^5(어림)": round(flaky["pass_rate"] ** TRIALS, 2)})
show(sweep, title="실수 확률별 반복 신뢰성 (모의)")

**관찰할 것.** 확률이 0이면 세 숫자가 모두 1.0입니다. 확률이 올라가면 한 회차 통과율은 조금씩만 내려가는데 `pass^5`는 훨씬 빨리 내려갑니다. 확률 0.1은 "열 번 중 한 번 실수"라서 얼핏 괜찮아 보이지만, 5회 모두 통과하는 사례 비율은 눈에 띄게 낮아집니다. `pass@5`는 거의 내려가지 않아서 이 문제를 가립니다. 같은 실행인데 어떤 숫자를 보고하느냐에 따라 "좋다"와 "위험하다"가 모두 가능합니다.

이 표의 숫자는 모의 실행의 산술입니다. 실제 에이전트가 사례마다 어느 정도 흔들리는지는 실제 모델로 반복해 봐야 알고, 그때도 표본이 작으면 구간이 넓습니다. 이 노트북은 그 구간을 계산하지 않습니다.

## 4. 낮은 점수와 평가 불능은 다른 말이다

판정이 `FAIL`인 것과, 판정할 수 없는 것은 전혀 다릅니다. `FAIL`은 "기록을 읽어 보니 치명 위반이 있다"는 **측정 결과**입니다. 반면 판정할 수 없는 경우는 측정 자체가 이루어지지 않은 것입니다. 이 둘을 한 통에 넣어 평균을 내면 숫자가 의미를 잃습니다. 궤적 판정(`TrajectoryVerdict.status`)은 다섯 가지이고, 앞 조건이 먼저 이깁니다.

| 상태 | 뜻 | 종류 |
|---|---|---|
| `INCOMPLETE` | 기록이 불완전하다. 읽은 기록에 근거 ID가 빠졌거나, 읽기 예산(사건 200개)을 넘었다 | 평가 불능 |
| `PROVISIONAL` | 실행이 오류로 끝났거나 중간에 잘려서 아직 끝나지 않았다. 끝난 뒤 다시 판정해야 한다 | 평가 불능 |
| `NO_EVIDENCE` | 도구 사건도 응답도 없어 판정할 근거가 없다 | 평가 불능 |
| `FAIL` | 치명 위반이 하나 이상 있다 | 낮은 점수 |
| `PASS` | 관측한 범위에서 치명 위반이 없다. 좋은 작업이라는 증명은 아니다 | 통과 |

한 가지 흔한 오해를 먼저 풉니다. **정상적으로 끝난 Episode가 마지막에 고객에게 추가 질문을 했다고 해서 평가 불능이 되지 않습니다.** 주문 번호가 없는 요청에서 "주문 번호를 알려 주세요"라고 되묻는 것이 정답인 사례가 있습니다. 그런 Episode는 끝난 것이므로 다른 답변과 똑같이 판정합니다. `PROVISIONAL`은 실행 상태가 `completed`가 아닐 때(오류, 중단) 또는 실행기가 고객의 답을 기다리며 멈췄다는 표시(`Artifact.awaiting_user`)가 있을 때입니다. 이 노트북의 실행기에서는 후자가 일어나지 않습니다.

**먼저 예측하세요.** 아래 일곱 줄에서 `PASS`, `FAIL`, `PROVISIONAL`, `INCOMPLETE`, `NO_EVIDENCE`는 각각 몇 줄일까요? 주문 번호가 없는 요청에 되묻고 끝난 Episode는 어느 칸일까요?

In [ ]:
# [따라하기] 다섯 가지 판정 상태를 모두 만든다: 실제 실행 네 개와 일부러 만든 기록 하나
missing_card = cards["missing-id"]
normal_again = run_episode(normal_card)

examples = [
    ("정상 흐름", normal_again, {}),
    ("승인 전 쓰기", run_episode(normal_card, driver=make_mistake_driver({"write_before_approval": 1.0}, seed=0)), {}),
    ("주문 번호 없음 -> 되묻고 끝남", run_episode(missing_card), {}),
    ("예산 오류로 시작도 못 함", run_episode(normal_card, max_turns=0), {}),
    ("호출자가 기록 불완전을 알림", normal_again, {"complete": False}),
    ("읽기 예산(3개) 초과", normal_again, {"read_budget": 3}),
    # 마지막 줄은 실제 실행이 아니라 만든 기록이다: 사건과 응답을 비운 사본.
    ("사건·응답이 모두 없음(만든 기록)", normal_again.model_copy(update={"events": [], "response": None}), {}),
]

KIND = {"PASS": "통과", "FAIL": "낮은 점수", "PROVISIONAL": "평가 불능", "INCOMPLETE": "평가 불능", "NO_EVIDENCE": "평가 불능"}
rows = []
for label, run, options in examples:
    verdict = evaluate_trajectory(run, **options)
    rows.append({"경우": label, "실행 상태": run.execution_status, "응답 유형": run.response.kind if run.response else "-",
                 "판정": verdict.status, "종류": KIND[verdict.status], "이유": verdict.reason})
show(rows, title="궤적 판정 다섯 가지")
print(Counter(row["판정"] for row in rows))

**관찰할 것.** 판정 칸에 다섯 가지가 모두 나옵니다. 세 번째 줄(되묻고 끝난 Episode)은 `PASS`입니다. 응답 유형이 `clarification`이어도 실행이 끝났고 치명 위반이 없기 때문입니다. 네 번째 줄은 도구도 부르지 못하고 끝난 실행이라 `PROVISIONAL`이고, 이 경우 실행 상태 칸에 `infra_error`가 보입니다. 오류로 멈춘 실행을 `FAIL`로 세면 "에이전트가 틀렸다"와 "실행이 깨졌다"가 섞입니다.

다섯째와 여섯째 줄은 같은 정상 실행인데 `INCOMPLETE`입니다. 기록을 믿을 수 없거나 너무 길어서 읽지 못했다는 판정이지 "위반이 있다"가 아닙니다. 그런 경우에도 `fatal` 칸에는 찾은 위반이 참고용으로 남습니다. 일곱째 줄은 실제 실행이 아니라 사건과 응답을 일부러 비워 만든 기록입니다. 운영 trace에서 가끔 보이는 "내용이 안 남은 기록"이 이런 모양입니다.

운영 중 만나는 되묻기 Episode를 한 번 더 자세히 봅니다. 아래 셀은 주문 번호가 없는 요청의 근거에서 실행 상태, 마지막 응답 유형, 고객 답 대기 표지를 읽습니다.

In [ ]:
# [따라하기] 되묻고 끝난 Episode의 근거: 끝난 실행이다
clarified = run_episode(missing_card)
print("실행 상태:", clarified.execution_status)
print("응답:", clarified.response.kind, "|", clarified.response.message)
print("고객 답 대기 표지(awaiting_user):", clarified.awaiting_user)
print("서비스가 받은 도구 호출 수:", len(clarified.events))
print("궤적 판정:", evaluate_trajectory(clarified).status)

### 평가 불능을 어떻게 세느냐에 따라 숫자가 바뀐다

평가 불능을 어떻게 다루느냐가 통과율을 바꿉니다. 아래 셀은 정상 실행 6개, 승인 전 쓰기 2개, 오류 2개로 열 개의 판정을 만들고 통과율을 세 가지 방식으로 셉니다. 어느 것이 틀렸다는 뜻이 아닙니다. 어느 숫자가 무엇을 숨기는지를 보는 것입니다.

In [ ]:
# [깨뜨리기] 평가 불능을 FAIL로 세거나 조용히 빼면 통과율이 어떻게 움직이는가
batch = []
for trial in range(1, 7):
    batch.append(run_episode(normal_card, trial=trial))
for seed in (1, 2):
    batch.append(run_episode(normal_card, driver=make_mistake_driver({"write_before_approval": 1.0}, seed=seed)))
for trial in (1, 2):
    batch.append(run_episode(normal_card, max_turns=0, trial=trial))

statuses = Counter(evaluate_trajectory(run).status for run in batch)
passed, failed = statuses["PASS"], statuses["FAIL"]
evaluable = passed + failed
total = len(batch)

show([
    {"세는 방식": "평가 불능을 FAIL로 센다", "통과율": round(passed / total, 2), "숨기는 것": "측정하지 못한 것을 실패로 단정함"},
    {"세는 방식": "평가 불능을 조용히 뺀다", "통과율": round(passed / evaluable, 2), "숨기는 것": "빠진 실행이 몇 개인지"},
    {"세는 방식": "통과율과 평가 완료율을 같이 싣는다", "통과율": round(passed / evaluable, 2),
     "숨기는 것": f"평가 완료율 {evaluable}/{total} = {evaluable / total:.2f}를 함께 적음"},
], title="같은 열 개 실행, 세 가지 보고")
print("상태별 개수:", dict(statuses))

**관찰할 것.** 같은 열 개의 실행에서 통과율이 0.6과 0.75로 갈립니다. 평가 불능을 FAIL로 세면 에이전트가 억울한 점수를 받고, 조용히 빼면 열 개 중 두 개가 사라진 것을 아무도 모릅니다. 세 번째 방식은 통과율 옆에 **평가 완료율**(통과와 실패의 합 / 전체)을 함께 적습니다. 완료율이 낮으면 통과율은 읽을 가치가 줄어듭니다. 이 저장소는 모니터링(04)에서도 평가 완료율을 따로 보고, 홀드아웃 판정(06)에서는 평가 완료율이 90% 미만이면 개선을 판정하지 않고 "판정 불가"로 보고합니다.

한 가지 더, 이 열 개는 모두 모의 실행입니다. "정상 6, 실수 2, 오류 2"는 제가 정한 구성이지 실제 운영 분포가 아닙니다.

## 직접 해보기

세 가지를 해 봅니다. 해보기 1은 기본으로 실행됩니다(모델 호출 없음). 해보기 2는 `RUN_EXERCISE`, 해보기 3은 `RUN_OPTIONAL`이 `True`일 때만 실행되고 기본값은 둘 다 `False`입니다. 스위치는 맨 앞 설정 셀에 있습니다. 켜기 전에 위의 호출 규모를 다시 읽어 보세요. 해보기 3은 유료입니다.

### 해보기 1. 내 실수 조합 만들기

`make_mistake_driver`의 실수는 다섯 가지입니다. `write_before_approval`, `blind_retry`, `claim_without_record`, `write_out_of_scope`, `skip_final_check`. 사례 하나와 실수 조합을 고르고, 실행하기 전에 **예측을 먼저 적으세요.** 궤적 판정이 `PASS`일지 `FAIL`일지, 어떤 규칙이 걸릴지. 아래 셀은 정상 실행과 내 조합을 같은 표에 놓고 예측과 비교합니다. `blind_retry`는 결제 TIMEOUT 사례(`timeout-before`, `timeout-after`)에서만 일어날 수 있고, `write_out_of_scope`는 다른 고객의 주문에 쓰기를 시도하는 실수입니다.

In [ ]:
# [내 것 만들기] 사례와 실수 조합을 골라 예측과 관찰을 비교한다 (모델 호출 없음)
MY_SCENARIO = "timeout-after"   # 바꿔보세요: cards의 사례 이름 중 하나
MY_MISTAKES = {"blind_retry": 1.0, "skip_final_check": 1.0}   # 바꿔보세요: 실수 이름과 확률(0~1)
MY_PREDICTION = "FAIL"          # 실행하기 전에 내가 예상한 궤적 판정(PASS, FAIL, PROVISIONAL 등)

my_card = cards[MY_SCENARIO]
baseline_my = run_episode(my_card, release="candidate")
mistaken_my = run_episode(my_card, release="candidate", driver=make_mistake_driver(MY_MISTAKES, seed=0))

rows = []
for label, run in (("정상 실행", baseline_my), ("내 실수 조합", mistaken_my)):
    verdict = evaluate_trajectory(run)
    rows.append({"실행": label, "응답 유형": run.response.kind if run.response else "-", "궤적 판정": verdict.status,
                 "치명 규칙": ", ".join(sorted({h.rule_id for h in verdict.fatal})) or "-",
                 "경고 규칙": ", ".join(sorted({h.rule_id for h in verdict.warn})) or "-",
                 "마지막 상태": state_summary(run.final_state)})
show(rows, title=f"{MY_SCENARIO}에 {MY_MISTAKES}를 넣은 결과")
print("내 예측:", MY_PREDICTION, "| 관찰:", rows[1]["궤적 판정"], "| 일치:", MY_PREDICTION == rows[1]["궤적 판정"])

**관찰할 것.** 예측이 틀렸다면 어느 가정이 틀렸는지 찾아 보세요. 흔한 오해는 세 가지입니다. 실수가 일어날 자리가 사례에 없으면(예: 주문 번호가 없는 사례에 `write_before_approval`) 드라이버가 끼워 넣지 못합니다. 등급이 경고인 규칙(`blind-retry`, `skip-final-check`)만 걸리면 판정은 `PASS`이고 경고 칸에만 남습니다. 치명 규칙이 하나 이상이어야 `FAIL`입니다. 그리고 한 가지 실수가 다른 규칙을 연쇄로 부르기도 합니다. 기본 조합에서도 경고 두 개 말고 치명 규칙 `claim-without-record`가 함께 걸렸는데, 완료를 호출한 뒤 `get_order`로 확인하지 않은 채 완료라고 보고했기 때문입니다. 경고 규칙을 만드는 실수가 완료 주장의 근거까지 없애는지 확인해 보세요.

### 해보기 2. 반복 횟수를 늘리기 (모델 호출 없음, 시간이 조금 더 듭니다)

3절의 모의 반복을 회차 20번으로 늘려서 `pass@20`과 `pass^20`을 봅니다. 모델 호출은 없지만 Episode가 이백 개 단위로 늘어납니다. 실수 확률 0.1에서도 `pass^20`이 얼마나 내려가는지 확인하세요.

In [ ]:
# [바꿔보기] 회차를 20번으로 늘려 pass@20과 pass^20을 본다 (RUN_EXERCISE가 True일 때만, 모델 호출 없음)
if RUN_EXERCISE:
    long_plan, long_rows = collect("baseline", 0.1, trials=20)
    steady_long_plan, steady_long_rows = collect("candidate", 0.0, trials=20)
    long_report = {"plan": long_plan + steady_long_plan, "rows": long_rows + steady_long_rows, "mode": "scripted"}
    long_summary = compare(long_report)["summaries"]
    show([{"묶음": name, "계획 수": s["planned"], "한 회차 통과율": round(s["pass_rate"], 2),
           "pass@20": s["pass_at_k"], "pass^20": s["pass_all_k"]} for name, s in long_summary.items()],
         title="회차 20번 (모의)")
else:
    print("RUN_EXERCISE가 False라서 건너뜁니다. 설정 셀에서 True로 바꾸면 실행됩니다.")

**관찰할 것.** 스위치를 켰다면, 한 회차 통과율은 3절과 비슷한 수준인데 `pass^20`은 훨씬 낮아졌을 것입니다. 스위치가 꺼져 있다면 건너뛰었다는 안내 한 줄만 나옵니다. 이 안내가 나오는 것도 정상입니다.

### 해보기 3. 실제 모델 Episode 하나 (선택, 유료)

지금까지는 모두 모의 실행이었습니다. 이제 실제 Claude 에이전트가 같은 정상 요청을 처리하게 해서, 같은 두 평가기로 채점합니다. 결과는 **보여 주기만** 하고 어떤 기대도 확인하지 않습니다. 모델은 매번 다르게 행동할 수 있고, 한 번의 관찰은 성능이 아닙니다. 에이전트 호출은 Episode당 최대 12회로 묶여 있으며(`MAX_MODEL_CALLS`), 이 실행에서 Jev와 Judge는 쓰지 않습니다.

In [ ]:
# [바꿔보기] 실제 Claude 에이전트로 정상 요청 Episode 하나를 실행하고 같은 평가기로 채점한다 (유료: RUN_OPTIONAL이 True일 때만)
if RUN_OPTIONAL:
    announce("01 실제 모델 Episode 1개", model_calls=12, minutes=1)
    live_langfuse = langfuse_or_none()
    live_model = lab_settings.make_agent_model(lab_settings.load_settings())
    live_run = execute_episode(
        normal_card.input, fixtures[normal_card.fixture_id], followups=normal_card.followups,
        max_turns=normal_card.max_turns, scenario_id="normal", release="baseline", trial=1,
        mode="live", prompt=PROMPTS["baseline"], model=live_model, langfuse=live_langfuse,
        environment_tag="day05_new", tags=["01-why-evaluate", "live"], version="live-baseline")
    if live_langfuse is not None:
        live_langfuse.flush()

    print("실행 상태:", live_run.execution_status, "| 모델 호출:", live_run.model_calls)
    print("답변:", live_run.response.kind if live_run.response else "-", "|", live_run.response.message if live_run.response else "-")
    show(trace_rows(live_run), title="도구 호출 순서")
    show(verdict_rows(live_run, normal_card), title="오프라인 평가기: 검사별 상태")
    live_verdict = evaluate_trajectory(live_run)
    print("궤적 판정:", live_verdict.status, "|", live_verdict.reason)
    (run_dir / "live_episode.json").write_text(
        json.dumps(episode_summary(live_run), ensure_ascii=False, indent=2), encoding="utf-8")
else:
    print("RUN_OPTIONAL이 False라서 건너뜁니다. 설정 셀에서 True로 바꾸면 실제 모델을 한 번 호출합니다(유료).")

**관찰할 것.** 스위치를 켰다면 모델이 같은 정상 요청에서 어떤 순서로 도구를 불렀는지 표로 읽어 보세요. 2절의 정상 궤적과 같은가요? 다르다면 어떤 규칙이 걸리는지, 아니면 순서가 달라도 `PASS`인지 확인합니다. 한 번 `PASS`가 나왔다고 모델이 늘 맞는다고 말할 수는 없습니다. 3절에서 본 대로 그 한 번이 `pass@k`의 한 칸일 뿐입니다. 꺼져 있다면 건너뛰었다는 안내만 나옵니다.

## 확인 질문

답을 말로 설명할 수 있는지 확인합니다. 정답을 적어 두지 않았습니다. 막히면 해당 절의 셀을 다시 실행해서 출력을 근거로 설명해 보세요.

1. 2절에서 정상 실행과 승인 전 쓰기 실행은 최종 답변과 마지막 업무 상태가 같았습니다. 그런데 왜 둘을 "같은 성공"이라고 부를 수 없는지, 어떤 근거(Artifact의 어느 부분)로 설명할 수 있나요? 오프라인 평가기에서는 어느 검사 하나만 갈렸는지도 함께 말해 보세요.
2. 3절에서 `pass@5`는 1.0에 가까운데 `pass^5`는 낮았습니다. 주문 취소 에이전트를 사람이 매번 확인하지 않고 운영한다면 둘 중 어느 숫자를 보고하겠습니까? 그리고 이 숫자들이 모의 실행에서 나왔다는 사실은 보고서에 어떻게 적어야 할까요?
3. 평가 불능(`INCOMPLETE`, `PROVISIONAL`, `NO_EVIDENCE`)을 `FAIL`로 세는 것과 조용히 빼는 것은 각각 무엇을 잘못 전달하나요? 평가 완료율을 통과율 옆에 적는 이유를 자기 말로 설명해 보세요.
4. 주문 번호가 없어서 되묻고 끝난 Episode는 왜 `PROVISIONAL`이 아니라 `PASS`가 되었나요? 어떤 조건이면 같은 Episode가 `PROVISIONAL`이 되나요?
5. 답변 유형만 보는 평가기는 실수를 넣든 안 넣든 같은 점수를 줬습니다. 에이전트를 개선했는데 이 평가기의 점수가 그대로라면 무엇을 알 수 있고 무엇을 알 수 없나요?

## 정리

- 평가는 세 질문으로 나눕니다. 결과(실제 상태), 과정(호출의 대상·권한·순서), 답변(전달 품질). 각각 근거가 다르므로 답변 한 줄 로그만으로는 둘을 복원할 수 없습니다.
- 같은 최종 답변이 서로 다른 궤적에서 나올 수 있고, 답변만 보는 평가기는 그 차이를 측정하지 못합니다. 이 노트북에서 승인 전 쓰기를 넣은 실행은 답변과 마지막 상태가 정상 실행과 같았지만 궤적 규칙은 `FAIL`과 첫 위반 사건을 줬습니다.
- 반복 실행에서는 `pass@k`(한 번이라도)와 `pass^k`(모두)가 다른 질문에 답합니다. 운영에 맡기는 에이전트는 `pass^k`가 중요하고, 둘 다 사례 단위로 읽어야 합니다. 이 노트북의 반복은 모두 모의 실행이었습니다.
- 낮은 점수(`FAIL`)와 평가 불능(`INCOMPLETE`, `PROVISIONAL`, `NO_EVIDENCE`)은 다른 말입니다. 끝난 Episode가 되묻고 끝난 것은 평가 불능이 아닙니다. 보고서에는 통과율과 평가 완료율을 함께 적습니다.
- 파일은 이 노트북의 작업 폴더(`outputs/notebook/01/` 아래 실행별 폴더)에만 썼습니다. 반복 요약(`repeated_summary.json`)과 선택 실행의 Episode 요약이 거기 있습니다.

### 미확인으로 남은 것

- 실제 Claude, Jev, Judge의 응답. 이 노트북의 정량 숫자는 모두 모의 실행이며, 실제 모델이 어떤 실수를 얼마나 자주 하는지는 확인되지 않았습니다.
- `ToolStrategy(Response)`가 `claude-sonnet-5-5`에서 구조화 응답을 안정적으로 내는지. 해보기 3의 첫 유료 실행에서 확인할 일입니다.
- scripted Episode가 실제 Langfuse 서버에서 trace 하나로 묶이는지. 오프라인 시험에서는 묶임을 확인했지만 실제 서버는 첫 실행에서 확인합니다.
- Langfuse 서버 쪽 동작(집계 지연, 2026-11-16 이후 Cloud 동작).

다음 노트북(02)은 "무엇을 성공으로 볼지"를 사람이 검수한 Golden Dataset으로 고정하는 방법입니다.